# CPD Combined Pipeline: PELT (ruptures) + ChangeForest

This notebook combines two change point detection approaches applied to MLB statcast hitter data:

- **Algorithm 1 — PELT (ruptures)**: Univariate mean-shift detection on four separate CPD indicators (hitting decisions, power efficiency, wOBA residual, launch angle stability) using the Pruned Exact Linear Time algorithm.
- **Algorithm 2 — ChangeForest**: Multivariate random-forest-based CPD on aligned power efficiency + wOBA residual features.

Both approaches follow the Truong et al. (2020) five-step CPD framework.

## CPD Framework (Truong et al., 2020)

- Step 1: Define the signal
- Step 2: Define the type of change (cost function)
- Step 3: Select the optimization/search method
- Step 4: Determine the number of change points (constraints / penalty)
- Step 5: Evaluate and validate the results

Before entering the pipeline, we first prepare the input data.

## 0. Imports & Data Loading

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D

import ruptures as rpt
from sklearn.preprocessing import StandardScaler
from changeforest import changeforest, Control

In [ ]:
# --- Load data ---
DATA_FILE = 'Qualified_hitters_statcast_2021_2025_pa_master.csv'
csv_path = Path(os.getcwd()).parent / 'data' / 'processed' / DATA_FILE
df = pd.read_csv(csv_path)

# --- Data availability summary for the four CPD indicators ---
CPD_INDICATORS = [
    'hitting_decisions_score',
    'power_efficiency',
    'woba_residual',
    'launch_angle_stability_50pa',
]

summary_df = pd.DataFrame({
    'indicator': CPD_INDICATORS,
    'rows_with_value': [df[col].notna().sum() for col in CPD_INDICATORS],
})
summary_df['pct_of_total'] = (summary_df['rows_with_value'] / len(df) * 100).round(2)

print(f"Total rows: {len(df):,}  |  Total batters: {df['batter'].nunique()}")
display(summary_df)

## Step 1: Define the Signal

Following Truong et al. (2020), the first step is to prepare the time series data by defining the signal, including the variable, time index, and preprocessing approach.

**Objective**  
Prepare time series data — what data will be analyzed?

**Decision Made**  
- **X**: sequential PA index (e.g., `pa_seq_id`)  
- **Y**: corresponding feature value, including:
  - Hitting Decisions  
  - Power Efficiency  
  - Luck vs. Skill (wOBA Residual)  
  - Launch Angle Stability  

We filter by player, remove missing values, and sort by the sequence index to ensure a properly ordered time series.

In [ ]:
def cpd_subdataset_generator(df, selected_player_id):
    """
    Generate four CPD subdatasets for one player.

    Parameters
    ----------
    df : pandas.DataFrame
        Original PA-level dataframe.
    selected_player_id : int
        MLBAM batter id.

    Returns
    -------
    dict[str, pandas.DataFrame]
        Keys: cpd_decision, cpd_power_efficiency,
              cpd_woba_residual, cpd_launch_angle_stability
    """
    base_cols = ["batter", "pa_uid", "game_date", "game_pk", "at_bat_number"]

    def _build_subdataset(x_col, y_col):
        return (
            df.loc[df["batter"] == selected_player_id, base_cols + [x_col, y_col]]
              .dropna(subset=[x_col, y_col])
              .sort_values(x_col)
              .reset_index(drop=True)
        )

    return {
        "cpd_decision": _build_subdataset("pa_seq_id", "hitting_decisions_score"),
        "cpd_power_efficiency": _build_subdataset("power_woba_seq_id", "power_efficiency"),
        "cpd_woba_residual": _build_subdataset("power_woba_seq_id", "woba_residual"),
        "cpd_launch_angle_stability": _build_subdataset("launch_angle_seq_id", "launch_angle_stability_50pa"),
    }


def cpd_subdataset_graph_generator(cpd_subdatasets, selected_player_id=None, figsize=(14, 10)):
    """
    Plot the 4 CPD subdatasets as a 2x2 panel.

    Parameters
    ----------
    cpd_subdatasets : dict[str, pandas.DataFrame]
        Output from cpd_subdataset_generator.
    selected_player_id : int | None
        Optional, used for figure title.
    figsize : tuple[int, int]
        Figure size for matplotlib.

    Returns
    -------
    (fig, axes)
        Matplotlib figure and axes array.
    """
    plot_specs = [
        ("cpd_decision", "pa_seq_id", "hitting_decisions_score", "CPD 1: Hitting Decisions"),
        ("cpd_power_efficiency", "power_woba_seq_id", "power_efficiency", "CPD 2: Power Efficiency"),
        ("cpd_woba_residual", "power_woba_seq_id", "woba_residual", "CPD 3: wOBA Residual"),
        (
            "cpd_launch_angle_stability",
            "launch_angle_seq_id",
            "launch_angle_stability_50pa",
            "CPD 4: Launch Angle Stability",
        ),
    ]

    fig, axes = plt.subplots(2, 2, figsize=figsize, constrained_layout=True)
    axes = axes.flatten()

    for ax, (key, x_col, y_col, title) in zip(axes, plot_specs):
        subdf = cpd_subdatasets.get(key)

        if subdf is None or subdf.empty:
            ax.set_title(f"{title} (no data)")
            ax.set_xlabel(x_col)
            ax.set_ylabel(y_col)
            ax.grid(alpha=0.3)
            continue

        ax.plot(subdf[x_col], subdf[y_col], linewidth=1.4, alpha=0.9)
        ax.scatter(subdf[x_col], subdf[y_col], s=8, alpha=0.45)
        ax.set_title(f"{title} (n={len(subdf)})")
        ax.set_xlabel(x_col)
        ax.set_ylabel(y_col)
        ax.grid(alpha=0.3)

    fig.suptitle(
        "CPD Indicator Subdatasets" if selected_player_id is None
        else f"CPD Indicator Subdatasets | Player ID: {selected_player_id}",
        fontsize=14,
    )

    return fig, axes


# Example usage
SELECTED_PLAYER_ID = 660271  # Shohei Ohtani
cpd_subdatasets = cpd_subdataset_generator(df, SELECTED_PLAYER_ID)

cpd_decision = cpd_subdatasets["cpd_decision"]
cpd_power_efficiency = cpd_subdatasets["cpd_power_efficiency"]
cpd_woba_residual = cpd_subdatasets["cpd_woba_residual"]
cpd_launch_angle_stability = cpd_subdatasets["cpd_launch_angle_stability"]

print({
    "cpd_decision_rows": len(cpd_decision),
    "cpd_power_efficiency_rows": len(cpd_power_efficiency),
    "cpd_woba_residual_rows": len(cpd_woba_residual),
    "cpd_launch_angle_stability_rows": len(cpd_launch_angle_stability),
})

fig, axes = cpd_subdataset_graph_generator(cpd_subdatasets, selected_player_id=SELECTED_PLAYER_ID)
plt.show()

---
# Algorithm 1: PELT (ruptures) — Univariate Mean-Shift Detection

Uses the **Pruned Exact Linear Time (PELT)** algorithm from the `ruptures` library to detect mean shifts in each of the four CPD indicators independently.

- **Cost function**: L2 (least squares), detects changes in the mean level of the signal
- **Search method**: PELT — provides an exact solution to the penalized segmentation problem, efficient for long time series
- **Constraint**: Penalty parameter `pen` controls the number of change points detected

### Step 2: Define the Type of Change (Cost Function)

Different cost functions capture different types of structural changes:

- **L2 (Least Squares)**: Detects changes in the mean level of the signal (mean shift).
- **RBF (Radial Basis Function)**: Captures more general distributional changes, including nonlinear patterns and variance shifts.
- **Normal Model**: Assumes a Gaussian distribution and detects changes in both mean and variance.

Given that some indicators exhibit relatively stable mean but changing variability, models beyond simple mean shift (e.g., RBF, Normal) are also considered.

### Step 3: Select the Optimization/Search Method

We adopt **PELT** (Pruned Exact Linear Time) as the search algorithm:

- Provides an exact solution to the penalized segmentation problem
- Efficiently identifies globally optimal change points
- Suitable for long time series (e.g., thousands of observations)

### Step 4: Determine Number of Changes (Penalty)

The number of detected change points is controlled by a penalty parameter (`pen`):

- **Lower penalty** → more change points (risk of overfitting)
- **Higher penalty** → fewer change points (risk of underfitting)

The penalty is tuned via grid search to balance sensitivity and interpretability.

In [ ]:
# Signal specification for all four indicators
CPD_SIGNAL_SPECS = {
    "cpd_decision": ("pa_seq_id", "hitting_decisions_score", "CPD 1: Hitting Decisions"),
    "cpd_power_efficiency": ("power_woba_seq_id", "power_efficiency", "CPD 2: Power Efficiency"),
    "cpd_woba_residual": ("power_woba_seq_id", "woba_residual", "CPD 3: wOBA Residual"),
    "cpd_launch_angle_stability": (
        "launch_angle_seq_id",
        "launch_angle_stability_50pa",
        "CPD 4: Launch Angle Stability",
    ),
}


def apply_mean_shift_smoothing(subdf, y_col, window=50, min_periods=1):
    """
    Apply rolling mean smoothing to a CPD subdataset.
    Returns the dataframe with an added '<y_col>_smoothed' column.
    """
    out = subdf.copy()
    out[f"{y_col}_smoothed"] = (
        out[y_col]
        .rolling(window=window, min_periods=min_periods)
        .mean()
    )
    return out


def build_cpd_signal(subdf, x_col, y_col):
    """
    Build univariate signal array (n, 1) for ruptures from a CPD subdataset.
    """
    signal_df = (
        subdf[[x_col, y_col]]
        .dropna()
        .sort_values(x_col)
        .reset_index(drop=True)
        .copy()
    )
    signal = signal_df[y_col].to_numpy().reshape(-1, 1)
    return signal_df, signal


def run_pelt_mean_shift(signal, pen=50, model="l2", min_size=50, jump=1):
    """
    Run mean-shift CPD using PELT.

    Parameters
    ----------
    signal : np.ndarray  shape (n, 1)
    pen    : float  penalty controlling number of breakpoints
    model  : str    ruptures cost model ('l2', 'rbf', 'normal')
    min_size : int  minimum segment length
    jump   : int    step between candidate breakpoints

    Returns
    -------
    list[int]  breakpoint indices (last element is n, not a real change point)
    """
    algo = rpt.Pelt(model=model, min_size=min_size, jump=jump).fit(signal)
    return algo.predict(pen=pen)

In [ ]:
def detect_mean_shift(subdf, x_col, y_col, model="rbf", pen=12, min_size=50, jump=1):
    """
    Detect mean shifts in a univariate CPD subdataset using PELT.
    Returns a dict with breakpoint indices, x-axis positions, and a summary table.
    """
    if subdf.empty:
        return {
            "breakpoints": [],
            "breakpoint_x": [],
            "summary": pd.DataFrame(columns=["breakpoint_index", x_col, y_col]),
        }

    signal = subdf[[y_col]].to_numpy()
    algo = rpt.Pelt(model=model, min_size=min_size, jump=jump).fit(signal)
    raw_breakpoints = algo.predict(pen=pen)

    breakpoints = [bp for bp in raw_breakpoints if bp < len(subdf)]
    breakpoint_x = [subdf.iloc[bp][x_col] for bp in breakpoints]

    summary = pd.DataFrame({
        "breakpoint_index": breakpoints,
        x_col: breakpoint_x,
        y_col: [subdf.iloc[bp][y_col] for bp in breakpoints],
    })

    return {
        "breakpoints": breakpoints,
        "breakpoint_x": breakpoint_x,
        "summary": summary,
    }


def plot_mean_shift_result(subdf, x_col, y_col, title, mean_shift_result, figsize=(12, 4)):
    """Plot one CPD signal with vertical lines at detected mean-shift breakpoints."""
    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(subdf[x_col], subdf[y_col], linewidth=1.4, alpha=0.9, label=y_col)
    ax.scatter(subdf[x_col], subdf[y_col], s=10, alpha=0.35)

    for i, x_value in enumerate(mean_shift_result["breakpoint_x"]):
        ax.axvline(
            x=x_value,
            color="crimson",
            linestyle="--",
            linewidth=1.2,
            alpha=0.85,
            label="detected mean shift" if i == 0 else None,
        )

    ax.set_title(title)
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.grid(alpha=0.3)
    ax.legend()
    return fig, ax


# --- Example: single indicator ---
SELECTED_SIGNAL_KEY = "cpd_decision"
x_col, y_col, title = CPD_SIGNAL_SPECS[SELECTED_SIGNAL_KEY]
selected_subdf = cpd_subdatasets[SELECTED_SIGNAL_KEY]

mean_shift_result = detect_mean_shift(
    selected_subdf,
    x_col=x_col,
    y_col=y_col,
    model="rbf",
    pen=40,
    min_size=50,
)

display(mean_shift_result["summary"])
plot_mean_shift_result(
    selected_subdf,
    x_col=x_col,
    y_col=y_col,
    title=f"{title} | Mean Shift Detection | Player ID: {SELECTED_PLAYER_ID}",
    mean_shift_result=mean_shift_result,
)
plt.show()

In [ ]:
def run_mean_shift_for_all_subdatasets(
    cpd_subdatasets,
    signal_specs,
    model="l2",
    pen=12,
    min_size=50,
    jump=1,
):
    """Run mean-shift detection for all four CPD subdatasets."""
    results = {}

    for key, (x_col, y_col, title) in signal_specs.items():
        subdf = cpd_subdatasets.get(key, pd.DataFrame())
        detection_result = detect_mean_shift(
            subdf,
            x_col=x_col,
            y_col=y_col,
            model=model,
            pen=pen,
            min_size=min_size,
            jump=jump,
        )
        results[key] = {
            "x_col": x_col,
            "y_col": y_col,
            "title": title,
            "subdf": subdf,
            "detection": detection_result,
        }

    return results


def plot_mean_shift_for_all_subdatasets(
    all_results,
    selected_player_id=None,
    figsize=(14, 10),
):
    """Plot mean-shift results for all four CPD subdatasets in a 2x2 panel."""
    fig, axes = plt.subplots(2, 2, figsize=figsize, constrained_layout=True)
    axes = axes.flatten()

    for ax, (key, result) in zip(axes, all_results.items()):
        subdf = result["subdf"]
        x_col = result["x_col"]
        y_col = result["y_col"]
        title = result["title"]
        breakpoint_x = result["detection"]["breakpoint_x"]

        if subdf.empty:
            ax.set_title(f"{title} (no data)")
            ax.set_xlabel(x_col)
            ax.set_ylabel(y_col)
            ax.grid(alpha=0.3)
            continue

        ax.plot(subdf[x_col], subdf[y_col], linewidth=1.3, alpha=0.9, label=y_col)
        ax.scatter(subdf[x_col], subdf[y_col], s=8, alpha=0.3)

        for i, x_value in enumerate(breakpoint_x):
            ax.axvline(
                x=x_value,
                color="crimson",
                linestyle="--",
                linewidth=1.1,
                alpha=0.8,
                label="detected mean shift" if i == 0 else None,
            )

        ax.set_title(f"{title} (n={len(subdf)}, bkps={len(breakpoint_x)})")
        ax.set_xlabel(x_col)
        ax.set_ylabel(y_col)
        ax.grid(alpha=0.3)
        ax.legend()

    figure_title = "Mean Shift Detection Across 4 CPD Indicators"
    if selected_player_id is not None:
        figure_title += f" | Player ID: {selected_player_id}"
    fig.suptitle(figure_title, fontsize=14)

    return fig, axes


all_mean_shift_results = run_mean_shift_for_all_subdatasets(
    cpd_subdatasets,
    CPD_SIGNAL_SPECS,
    model="l2",
    pen=50,
    min_size=50,
)

all_mean_shift_summary = pd.concat(
    [
        result["detection"]["summary"].assign(indicator=key)
        for key, result in all_mean_shift_results.items()
    ],
    ignore_index=True,
)

summary_counts = pd.DataFrame({
    "indicator": list(all_mean_shift_results.keys()),
    "n_rows": [len(result["subdf"]) for result in all_mean_shift_results.values()],
    "n_breakpoints": [len(result["detection"]["breakpoints"]) for result in all_mean_shift_results.values()],
})

display(summary_counts)
display(all_mean_shift_summary.head(20))

fig, axes = plot_mean_shift_for_all_subdatasets(
    all_mean_shift_results,
    selected_player_id=SELECTED_PLAYER_ID,
)
plt.show()

In [ ]:
def plot_cp_result(signal_df, x_col, raw_y_col, smooth_y_col, bkpts, title="CPD Result"):
    """Plot raw signal, smoothed signal, and detected breakpoints."""
    fig, ax = plt.subplots(figsize=(14, 5))

    ax.plot(signal_df[x_col], signal_df[raw_y_col], linewidth=1.0, alpha=0.30, label="Raw signal")
    ax.plot(signal_df[x_col], signal_df[smooth_y_col], linewidth=2.0, label="Smoothed signal")

    for bp in bkpts[:-1]:
        ax.axvline(signal_df.iloc[bp - 1][x_col], linestyle="--", alpha=0.8)

    ax.set_title(title)
    ax.set_xlabel(x_col)
    ax.set_ylabel(raw_y_col)
    ax.grid(alpha=0.3)
    ax.legend()
    plt.show()


# --- Smoothed pipeline example: Hitting Decisions ---
WINDOW = 100
PENALTY = 50

cpd_decision_smooth = apply_mean_shift_smoothing(
    cpd_decision,
    y_col="hitting_decisions_score",
    window=WINDOW,
    min_periods=1,
)

decision_signal_df, decision_signal = build_cpd_signal(
    cpd_decision_smooth,
    x_col="pa_seq_id",
    y_col="hitting_decisions_score_smoothed",
)

decision_bkpts = run_pelt_mean_shift(decision_signal, pen=PENALTY)
decision_internal_bkpts = decision_bkpts[:-1]

print("decision_bkpts:", decision_bkpts)
print("# of internal change points:", len(decision_internal_bkpts))

plot_df = cpd_decision_smooth[
    ["pa_seq_id", "hitting_decisions_score", "hitting_decisions_score_smoothed"]
].dropna().reset_index(drop=True)

plot_cp_result(
    plot_df,
    x_col="pa_seq_id",
    raw_y_col="hitting_decisions_score",
    smooth_y_col="hitting_decisions_score_smoothed",
    bkpts=decision_bkpts,
    title=f"Hitting Decisions | Smoothed Mean Shift | window={WINDOW}, pen={PENALTY}",
)

In [ ]:
# --- Grid search on penalty (sensitivity analysis) ---
print("Penalty grid search on smoothed Hitting Decisions signal:")
for pen in [10, 20, 50, 100, 200, 500, 1000]:
    bkpts = run_pelt_mean_shift(decision_signal, pen=pen)
    print(f"  pen={pen:5d}, #internal_bkpts={len(bkpts) - 1}")

### Step 5: Evaluate & Validate (PELT)

The detected change points are evaluated using:

- **Visual inspection**: Check whether change points align with visible shifts in the smoothed signal.
- **Sensitivity analysis**: Assess robustness with respect to rolling window size and penalty parameter (grid search above).
- **Domain interpretation**: Verify whether detected regimes correspond to meaningful changes in player performance or behavior.

---
# Algorithm 2: ChangeForest — Multivariate Random Forest CPD

Uses the `changeforest` library (random-forest-based binary segmentation) to detect change points in a **joint** multivariate signal of `power_efficiency` + `woba_residual`, capturing distributional changes that univariate approaches may miss.

**Key differences vs. PELT:**
| | PELT | ChangeForest |
|---|---|---|
| Signal | Univariate (one indicator at a time) | Multivariate (joint 2-feature) |
| Change type | Mean shift | General distributional change |
| Algorithm | Exact penalized segmentation | Random forest binary segmentation |
| Constraint | Penalty `pen` | `minimal_relative_segment_length` + `model_selection_alpha` |

### Step 0: Data Preparation for ChangeForest

ChangeForest requires a single aligned multivariate array. We build it by:

1. Filtering by player and keeping only rows with both `power_efficiency` and `woba_residual` non-null.
2. Optionally applying a rolling mean to smooth both features.
3. Standardizing the features before fitting.

In [ ]:
def changeforest_subdataset_generator(df, selected_player_id, window=50, min_periods=1):
    """
    Generate one aligned multivariate CPD subdataset for ChangeForest
    using power_efficiency + woba_residual.
    """
    base_cols = ["batter", "pa_uid", "game_date", "game_pk", "at_bat_number"]
    feature_cols = ["power_woba_seq_id", "power_efficiency", "woba_residual"]

    subdf = (
        df.loc[df["batter"] == selected_player_id, base_cols + feature_cols]
          .dropna(subset=["power_woba_seq_id", "power_efficiency", "woba_residual"])
          .sort_values("power_woba_seq_id")
          .reset_index(drop=True)
    )

    subdf[f"power_efficiency_rollmean_{window}"] = (
        subdf["power_efficiency"].rolling(window=window, min_periods=min_periods).mean()
    )
    subdf[f"woba_residual_rollmean_{window}"] = (
        subdf["woba_residual"].rolling(window=window, min_periods=min_periods).mean()
    )

    return subdf


def changeforest_subdataset_graph_generator(subdf, selected_player_id=None, window=50, figsize=(14, 5)):
    """
    Plot the 2 ChangeForest indicators as a 1x2 panel,
    showing raw signal and rolling mean.
    """
    plot_specs = [
        ("power_woba_seq_id", "power_efficiency", "Power Efficiency"),
        ("power_woba_seq_id", "woba_residual", "wOBA Residual"),
    ]

    fig, axes = plt.subplots(1, 2, figsize=figsize, constrained_layout=True)

    for ax, (x_col, y_col, title) in zip(axes, plot_specs):
        if subdf is None or subdf.empty:
            ax.set_title(f"{title} (no data)")
            ax.set_xlabel(x_col)
            ax.set_ylabel(y_col)
            ax.grid(alpha=0.3)
            continue

        smooth_col = f"{y_col}_rollmean_{window}"

        ax.plot(subdf[x_col], subdf[y_col], linewidth=1.0, alpha=0.25,
                color="steelblue", label="raw")
        ax.scatter(subdf[x_col], subdf[y_col], s=6, alpha=0.3,
                   color="steelblue")

        if smooth_col in subdf.columns:
            ax.plot(subdf[x_col], subdf[smooth_col], linewidth=2.0,
                    color="orange", label=f"rolling mean (w={window})")

        ax.set_title(f"{title} (n={len(subdf)})")
        ax.set_xlabel(x_col)
        ax.set_ylabel(y_col)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

    fig.suptitle(
        "ChangeForest Input Subdataset"
        if selected_player_id is None
        else f"ChangeForest Input Subdataset | Player ID: {selected_player_id}",
        fontsize=14,
    )

    return fig, axes


# Example usage
CF_PLAYER_ID = 592450
subdf_cf = changeforest_subdataset_generator(df, selected_player_id=CF_PLAYER_ID, window=50)
fig, axes = changeforest_subdataset_graph_generator(subdf_cf, selected_player_id=CF_PLAYER_ID, window=50)
plt.show()

In [ ]:
def run_changeforest(
    subdf,
    window=50,
    use_rollmean=True,
    standardize=True,
    minimal_relative_segment_length=0.05,
):
    """
    Run ChangeForest on one player's aligned 2-feature dataset.

    Fixed: model_selection_alpha = 0.02

    Parameters
    ----------
    subdf : pd.DataFrame  output of changeforest_subdataset_generator
    window : int          rolling window used when building subdf
    use_rollmean : bool   use smoothed features instead of raw
    standardize : bool    standardize features before fitting
    minimal_relative_segment_length : float  min segment length as fraction of n

    Returns
    -------
    result, cps, X_used, feature_names
    """
    if subdf is None or subdf.empty:
        return None, [], None, None

    if use_rollmean:
        feature_names = [
            f"power_efficiency_rollmean_{window}",
            f"woba_residual_rollmean_{window}",
        ]
    else:
        feature_names = ["power_efficiency", "woba_residual"]

    X_used = subdf[feature_names].copy().values

    if standardize:
        X_used = StandardScaler().fit_transform(X_used)

    control = Control(
        model_selection_alpha=0.02,
        minimal_relative_segment_length=minimal_relative_segment_length,
    )

    result = changeforest(
        X_used,
        method="random_forest",
        control=control
    )

    cps = result.split_points()

    return result, cps, X_used, feature_names


def plot_changeforest_result(
    subdf,
    cps,
    window=50,
    use_rollmean=True,
    selected_player_id=None,
    minimal_relative_segment_length=None,
    figsize=(14, 5)
):
    """
    Plot ChangeForest result using game_date as X-axis (human-readable).
    """
    if use_rollmean:
        y_cols = [
            f"power_efficiency_rollmean_{window}",
            f"woba_residual_rollmean_{window}",
        ]
    else:
        y_cols = ["power_efficiency", "woba_residual"]

    titles = ["Power Efficiency", "wOBA Residual"]
    dates = pd.to_datetime(subdf["game_date"])

    fig, axes = plt.subplots(1, 2, figsize=figsize, constrained_layout=True)

    for ax, y_col, title in zip(axes, y_cols, titles):
        ax.plot(dates, subdf[y_col], linewidth=1.8)

        for cp in cps:
            if 0 <= cp < len(subdf):
                cp_date = dates.iloc[cp]
                ax.axvline(x=cp_date, color="red", linestyle="--", alpha=0.8)

        ax.set_title(title)
        ax.set_xlabel("Game Date")
        ax.set_ylabel(y_col)
        ax.grid(alpha=0.3)

        ax.xaxis.set_major_locator(mdates.AutoDateLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
        ax.tick_params(axis="x", rotation=45)

    main_title = "ChangeForest Result | alpha=0.02"
    if selected_player_id is not None:
        main_title += f" | Player ID: {selected_player_id}"
    if minimal_relative_segment_length is not None:
        main_title += f" | min_rel_seg_len={minimal_relative_segment_length}"
    main_title += f" | window={window}"

    fig.suptitle(main_title, fontsize=14)

    return fig, axes

In [ ]:
# --- Run ChangeForest on one player ---
window = 50
min_rel_seg_len = 0.05

subdf_cf = changeforest_subdataset_generator(
    df,
    selected_player_id=CF_PLAYER_ID,
    window=window
)

result, cps, X_used, feature_names = run_changeforest(
    subdf_cf,
    window=window,
    use_rollmean=True,
    standardize=True,
    minimal_relative_segment_length=min_rel_seg_len
)

print("Features used:", feature_names)
print("Detected change points (no.):", len(cps))
print("Detected change points:", cps)

if len(cps) > 0:
    print(subdf_cf.iloc[cps][["power_woba_seq_id", "game_date", "pa_uid"]])

fig, axes = plot_changeforest_result(
    subdf_cf,
    cps,
    window=window,
    use_rollmean=True,
    selected_player_id=CF_PLAYER_ID,
    minimal_relative_segment_length=min_rel_seg_len
)
plt.show()

In [ ]:
# --- Sensitivity analysis: grid search over window x min_rel_seg_len ---
print("ChangeForest sensitivity analysis (window x min_rel_seg_len):")
for w in [30, 50, 80]:
    for min_seg in [0.01, 0.05, 0.10]:
        subdf_w = changeforest_subdataset_generator(df, selected_player_id=CF_PLAYER_ID, window=w)

        _, cps_grid, _, _ = run_changeforest(
            subdf_w,
            window=w,
            minimal_relative_segment_length=min_seg
        )

        print(
            f"  window={w:3d}, min_rel_seg_len={min_seg:.2f}"
            f" -> #change_points: {len(cps_grid)}"
        )

### Step 5: Evaluate & Validate (ChangeForest)

- **Visual inspection**: Change points plotted as vertical red dashed lines on the smoothed signals.
- **Sensitivity analysis**: Grid search over `window` and `minimal_relative_segment_length` shows how stable the detected change points are.
- **Comparison with PELT**: Cross-reference ChangeForest change points (multivariate) against PELT results (univariate per indicator) to identify consistently detected structural breaks.